# <font color="#418FDE" size="6.5" uppercase>**Bias Mitigation Techniques and Case Study**</font>

>Last update: 20250117.
    
By the end of this Lecture, you will be able to:
- Apply bias detection and mitigation strategies to AI models.
- Analyze leading practices and trade-offs in promoting fairness.
- Examine a real-world bias and fairness case study.


## **1. Comprehensive Bias Detection Techniques**

### **1.1. Data Patterns and Anomalies**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20B/image_01_01.png" width="250">



>* Identify biases through data pattern analysis.
* Adjust datasets to ensure fairer outcomes.

>* Use statistics to identify data patterns.
* Apply clustering to reveal hidden biases.

>* Hiring algorithm shows gender bias in data.
* Predictive policing reveals racial bias patterns.



In [ ]:
#@title Python Code - Data Patterns and Anomalies

# This code loads and preprocesses the UCI Adult dataset by handling missing values and encoding features.
# It visualizes data patterns with histograms, count plots, and a correlation heatmap.
# It evaluates fairness metrics and trains a logistic regression model to detect patterns, anomalies, and biases.

# Importing necessary libraries (uncomment if needed)
# !pip install --upgrade pip
# !pip install --upgrade aif360 fairlearn pandas numpy matplotlib seaborn scikit-learn
# !pip install 'aif360[inFairness]'
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# For fairness metrics and data auditing
from aif360.datasets import StandardDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate
# For machine learning model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# 1. Data Loading and Preprocessing
# Load the UCI Adult dataset from a URL
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]
data = pd.read_csv(url, header=None, names=column_names, na_values=' ?', sep=',\s', engine='python')
# Display the first few rows
print("First 5 rows of the dataset:")
print(data.head())
# Handle missing values by dropping rows with any missing values
data.dropna(inplace=True)

# Encode categorical variables using Label Encoding for simplicity
label_encoders = {}
categorical_columns = data.select_dtypes(include=['object']).columns.tolist()
categorical_columns.remove('income')  # Exclude the target variable
for col in categorical_columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

# Map for labeling
label_map = {'<=50K': 0, '>50K': 1}
# Encode 'income' using label_map
data['income'] = data['income'].map(label_map)
# Display the shape after preprocessing
print("\nShape after preprocessing:", data.shape)

# 2. Data Auditing
# Define sensitive attribute
sensitive_attr = 'sex'
# Initialize AIF360 StandardDataset
dataset = StandardDataset(
    df=data,
    label_name='income',
    favorable_classes=[1],  # Use 1 as the favorable label
    protected_attribute_names=['sex'],  # Specify the sensitive attribute
    privileged_classes=[[1]],  # Privileged group: Male (encoded as 1)
    categorical_features=['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'native-country'],
    features_to_drop=['fnlwgt']  # Drop irrelevant features
)
# Compute overall dataset metrics
dataset_metric = BinaryLabelDatasetMetric(dataset,
                                          privileged_groups=[{'sex': 1}],
                                          unprivileged_groups=[{'sex': 0}])
# Compute overall dataset metrics
print("\nDataset Metrics:")
print(f"Number of instances: {dataset_metric.num_instances()}")
print(f"Number of privileged instances: {dataset_metric.num_positives(privileged=True)}")
print(f"Number of unprivileged instances: {dataset_metric.num_positives(privileged=False)}")
print(f"Percentage of unprivileged instances: {dataset_metric.num_positives(privileged=False) / dataset_metric.num_instances() * 100:.2f} %")
print(f"Disparate Impact (privileged / unprivileged): {dataset_metric.disparate_impact():.2f}")

# 3. Exploratory Data Analysis (EDA)
# Set global Seaborn context for font sizes
sns.set_context("notebook", font_scale=0.7)
# Create a figure with custom subplot widths
fig, axes = plt.subplots(1, 3, figsize=(15, 3),
                         gridspec_kw={'width_ratios': [1, 1, 2]})  # First two plots are narrower, third plot is wider
# Plot 1: Age Distribution by Sex
sns.histplot(data, x='age', hue='sex', kde=True, bins=30, palette='Set1', ax=axes[0])
axes[0].set_title('Age Distribution by Sex')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend(labels=['Female', 'Male'])
# Plot 2: Income Distribution by Sex
sns.countplot(x='income', hue='sex', data=data, palette='Set2', ax=axes[1])
axes[1].set_title('Income Distribution by Sex')
axes[1].set_xlabel('Income >50K')
axes[1].set_ylabel('Count')
axes[1].legend(labels=['Female', 'Male'])
# Plot 3: Correlation Heatmap
sns.heatmap(data.corr(), annot=True, fmt=".2f", cmap='coolwarm', ax=axes[2])
axes[2].set_title('Correlation Heatmap')
# Adjust layout to prevent overlap
plt.tight_layout()
# Show the combined plots
plt.show()

# 4. Fairness Metrics Computation
# Split the dataset into training and testing
X = data.drop('income', axis=1)
y = data['income']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
# Scale numerical features
scaler = StandardScaler()
numerical_cols = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

# Train a Logistic Regression model
model = LogisticRegression(solver='liblinear')
model.fit(X_train, y_train)
# Predictions
y_pred = model.predict(X_test)
# Add predictions to the test set
test_data = X_test.copy()
test_data['income'] = y_test
test_data['prediction'] = y_pred

# Define privileged and unprivileged groups
privileged_groups = [{'sex': 1}]
unprivileged_groups = [{'sex': 0}]
# Convert to AIF360 dataset
aif360_test = StandardDataset(
    df=test_data,
    label_name='income',
    favorable_classes=[1],
    protected_attribute_names=['sex'],
    privileged_classes=[[1]],
    categorical_features=['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'native-country'],
    features_to_drop=['fnlwgt']
)

# Add predictions to the AIF360 dataset
aif360_test.labels = test_data['income'].values.reshape(-1,1)
aif360_test.predictions = test_data['prediction'].values.reshape(-1,1)
# Compute fairness metrics using AIF360
classified_metric = ClassificationMetric(
    aif360_test, aif360_test,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)
disparate_impact = dataset_metric.disparate_impact()
print(f"\nDisparate Impact (AIF360): {disparate_impact:.2f}")


### **1.2. Unveiling Hidden Biases through Algorithmic Audits**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20B/image_01_02.png" width="250">



>* Algorithmic audits identify hidden AI biases.
* Audits enhance fairness in hiring algorithms.

>* Assess bias impact on demographic groups.
* Implement corrective measures for fairness.

>* Uncover biases from complex variable interactions.
* Ensure AI systems align with ethical standards.



In [ ]:
#@title Python Code - In-Depth Algorithmic Audit on the Adult Dataset

# This code demonstrates a comprehensive approach to performing an algorithmic audit
# on the UCI Adult (Income) dataset. We'll check for biases along protected attributes
# such as race and gender. We'll use the Fairlearn library to compute fairness metrics
# and demonstrate how one might mitigate bias with a reweighing technique.

# Importing necessary libraries (uncomment if needed)
# !pip install 'aif360[inFairness]' fairlearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
# Fairlearn imports
from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate, false_negative_rate
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.reductions import ExponentiatedGradient, DemographicParity, ErrorRate
from fairlearn.reductions import GridSearch, DemographicParity, ErrorRate

# 1. Load the dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
    'hours-per-week', 'native-country', 'income']
df = pd.read_csv(
    url,
    header=None,
    names=column_names,
    na_values=' ?',
    skipinitialspace=True)
# Drop rows with missing values
df.dropna(inplace=True)
# For clarity, rename columns
df.rename(columns={'income':'target'}, inplace=True)
# Convert target to binary: 1 if >50K, 0 otherwise
df['target'] = df['target'].apply(lambda x: 1 if x.strip() == '>50K' else 0)
# Protected attributes
protected_features = ['sex', 'race']

# 2. Basic Preprocessing
# Let's convert all categorical variables (besides target) into dummies
# but we'll keep sex and race as separate columns for fairness analysis
df_enc = pd.get_dummies(
    df.drop(columns=['target']),
    drop_first=True
)
X = df_enc
y = df['target']
# For convenience, let's keep the original 'sex' and 'race' columns in separate arrays
# (These columns are not one-hot encoded in X because we used get_dummies with drop_first).
# We'll use them to analyze fairness specifically.
A_sex = df['sex'].apply(lambda x: 1 if x.strip() == 'Male' else 0)  # 1=Male, 0=Female
A_race = df['race'].apply(lambda x: x.strip())

# 3. Split into train & test
X_train, X_test, y_train, y_test, A_sex_train, A_sex_test, A_race_train, A_race_test = train_test_split(
    X, y, A_sex, A_race, test_size=0.2, random_state=42)

# 4. Train a basic Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. Predictions and baseline metrics
y_pred = model.predict(X_test)
print("=== Baseline Model Performance ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# 6. Fairness Metrics using Fairlearn's MetricFrame
# We'll compute selection rate, false positive rate, false negative rate
# across subgroups for 'sex' and 'race' separately.
metrics = {
    'selection_rate': selection_rate,
    'false_positive_rate': false_positive_rate,
    'false_negative_rate': false_negative_rate}

print("\n=== Fairness Metrics by Sex ===")
metric_frame_sex = MetricFrame(
    metrics=metrics,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=A_sex_test)
print(metric_frame_sex.by_group)

print("\n=== Fairness Metrics by Race ===")
metric_frame_race = MetricFrame(
    metrics=metrics,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=A_race_test)
print(metric_frame_race.by_group)

# We can also look at the difference between the subgroups (max-min).
print("\n=== Difference in metrics (max-min) for Sex ===")
print(metric_frame_sex.difference(method='between_groups'))
print("\n=== Difference in metrics (max-min) for Race ===")
print(metric_frame_race.difference(method='between_groups'))

# 7. Mitigation Example: Reweighing
# We'll use Fairlearn's GridSearch or ExponentiatedGradient with DemographicParity
# to find a model that better balances the selection rate across protected groups.
# For demonstration, let's mitigate with respect to 'sex' only.
# We'll define a function that trains a simple RF for the fairness estimator
def learner_fn(**kwargs):
    return RandomForestClassifier(random_state=42, **kwargs)

# Apply GridSearch to mitigate bias with respect to 'sex'
# Define fairness constraint
constraint = DemographicParity()

# Define the mitigator using GridSearch
mitigator = GridSearch(
    estimator=RandomForestClassifier(random_state=42),  # Classifier to use
    constraints=constraint,  # Fairness constraint
    grid_size=5  # Number of hyperparameter configurations to try
)

# Fit the mitigator on training data
mitigator.fit(X_train, y_train, sensitive_features=A_sex_train)

# Retrieve all predictors from the grid search
all_predictors = mitigator.predictors_  # List of all trained models

# Find the best model based on accuracy or another metric
best_classifier = None
best_accuracy = 0
for predictor in all_predictors:
    # Evaluate each model on the test set
    y_pred_temp = predictor.predict(X_test)
    accuracy_temp = accuracy_score(y_test, y_pred_temp)

    # Keep track of the best-performing model
    if accuracy_temp > best_accuracy:
        best_accuracy = accuracy_temp
        best_classifier = predictor

# Use the best classifier for predictions
y_pred_mitigated = best_classifier.predict(X_test)

# Evaluate the mitigated model
print("\n=== Mitigated Model Performance (w.r.t Sex Demographic Parity) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_mitigated))

# Fairness metrics after mitigation
metric_frame_sex_mitigated = MetricFrame(
    metrics=metrics,
    y_true=y_test,
    y_pred=y_pred_mitigated,
    sensitive_features=A_sex_test)

print("Fairness Metrics by Sex (Mitigated):")
print(metric_frame_sex_mitigated.by_group)
print("\nDifference (max-min) in metrics after mitigation:")
print(metric_frame_sex_mitigated.difference(method='between_groups'))

# 8. Conclusion
print("""
============================================================
SUMMARY:
- Baseline model performance and fairness metrics were computed.
- We found potential disparities in selection rate / FPR / FNR across different sexes/races.
- We performed a simple mitigation approach (GridSearch + DemographicParity).
- You can compare the 'before' and 'after' fairness metrics to assess improvements.
============================================================
""")


### **1.3. Real-Time Bias Detection Systems**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20B/image_01_03.png" width="250">



>* Ensures fairness by monitoring AI outputs.
* Builds trust through transparency and accountability.

>* Sophisticated algorithms analyze data for bias.
* Real-time adjustments promote inclusivity and equity.

>* Ethical concerns in data privacy and storage.
* Need for transparent and ethical data practices.



In [ ]:
#@title Python Code - Real-Time Bias Detection Systems

# This code demonstrates a simple real-time bias detection system using the Adult Income dataset.
# The goal is to predict whether an individual earns more than $50K based on various features,
# while continuously monitoring for bias in predictions across gender in "mini-batches" (simulating real-time streaming).
# We will use a Random Forest classifier and evaluate the model's performance to check for bias in predictions.

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# 1. Data Loading and Preprocessing
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
    'hours-per-week', 'native-country', 'income']
data = pd.read_csv(
    url, header=None, names=column_names, na_values=' ?', skipinitialspace=True)

# Drop rows with missing values
data.dropna(inplace=True)

# Encode categorical variables
categorical_columns = [
    'workclass', 'education', 'marital-status', 'occupation', 'relationship',
    'race', 'sex', 'native-country', 'income']
label_encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

# Separate features (X) and target (y)
X = data.drop('income', axis=1)
y = data['income']

# 2. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# 3. Model Training
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Model Evaluation (Overall)
y_pred = model.predict(X_test)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 5. Bias Analysis (Static)
# Check for bias in predictions based on gender
results = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred,
    'Sex': X_test['sex'].values
})

# Calculate the proportion of positive (1) predictions for each gender
bias_analysis = results.groupby('Sex')['Predicted'].mean()
print("\nBias Analysis (Proportion of Positive Predictions by Gender):")
print(bias_analysis)

# 6. Real-Time / Streaming Bias Detection Simulation
print("\n=== Real-Time Bias Detection Simulation ===")
batch_size = 100  # Simulate processing data in batches of 100
current_start = 0

# Convert X_test and y_test to NumPy arrays to iterate by index
X_test_np = X_test.values
y_test_np = y_test.values

# Tracking arrays to store predictions for each batch
stream_predictions = []
stream_genders = []

while current_start < len(X_test_np):
    batch_end = min(current_start + batch_size, len(X_test_np))
    batch_data = X_test_np[current_start:batch_end]
    batch_genders = batch_data[:, list(X_test.columns).index('sex')]

    # Make predictions for the batch
    batch_preds = model.predict(batch_data)

    # Store for analysis
    stream_predictions.extend(batch_preds)
    stream_genders.extend(batch_genders)

    # Compute simple "on-the-fly" bias check
    # We'll see if the difference in positive prediction rates between genders is > 10% in this batch
    batch_df = pd.DataFrame({
        'Predicted': batch_preds,
        'Sex': batch_genders
    })

    # Mean of 'Predicted' for each gender in the current batch
    mean_by_gender = batch_df.groupby('Sex')['Predicted'].mean()

    # If we have two genders in the batch, check difference
    # Note: "Sex" is label-encoded, so we assume 0 = Female, 1 = Male, depending on how the encoding happened
    # If there's only one gender in the batch (unlikely in a large dataset), skip the check
    if len(mean_by_gender) == 2:
        diff = abs(mean_by_gender.iloc[0] - mean_by_gender.iloc[1])
        print(f"Batch {current_start // batch_size + 1}: Mean Prediction Rates by Gender = {mean_by_gender.to_dict()}, Difference = {diff:.2f}")
        if diff > 0.10:
            print(f"  --> Potential Bias Detected in Batch {current_start // batch_size + 1} (Difference > 10%)")

    current_start = batch_end

print("\nReal-time simulation completed. Above, we flagged any batch if the difference in positive prediction rates between genders exceeded 10%.")


### **1.4. Interactive Graphical Analysis**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20B/image_01_04.png" width="250">



>* Visual tools highlight potential data biases.
* Facilitates stakeholder communication and transparency.

>* Dynamic interface for real-time bias mitigation.
* Continuous monitoring ensures fairness over time.

>* Promotes inclusivity and stakeholder engagement.
* Empowers equitable AI through transparency.



In [ ]:
#@title Python Code - Interactive Graphical Analysis in One Cell

# This code demonstrates an interactive graphical analysis using a Random Forest Classifier on the Breast Cancer dataset.
# It trains the model to predict whether a tumor is malignant or benign, evaluates its performance using a classification report and confusion matrix, and visualizes the model’s feature importances using a bar plot.
# Additionally, it creates an interactive scatter plot to explore the relationship between two features (mean radius and mean texture), highlighting patterns in the data and enabling deeper insights into the dataset’s structure.

# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import plotly.express as px

# Load the Breast Cancer dataset
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42)

# Create and train the Random Forest classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Generate predictions
predictions = model.predict(X_test)

# Print classification report and confusion matrix
print("Classification Report:")
print(classification_report(y_test, predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

# Feature importances
importances = model.feature_importances_
feature_importances = pd.Series(importances, index=X.columns)
feature_importances = feature_importances.sort_values(ascending=False)

# Plot feature importances (static barplot)
plt.figure(figsize=(7, 5))
sns.barplot(x=feature_importances, y=feature_importances.index)
plt.title('Feature Importances in Random Forest Classifier')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.show()

# Create the interactive scatter plot (using Plotly)
fig_scatter = px.scatter(
    X,
    x='mean radius',
    y='mean texture',
    color=y.astype(str),  # Convert numeric labels to string for better legend display
    labels={'color': 'Diagnosis (1 = Malignant, 0 = Benign)'},
    title='Interactive Scatter Plot: Mean Radius vs. Mean Texture'
)
fig_scatter.update_layout(
    width=800,  # Set the desired width (e.g., 800 pixels)
    height = 300,
    legend=dict(
        orientation="h",  # Horizontal legend
        yanchor="bottom",
        y=-0.3,  # Position legend below the plot
        xanchor="center",
        x=0.5
    )
)

# Show the figure
fig_scatter.show()


### **1.5. Navigating Ethical Challenges in Bias Exposure**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20B/image_01_05.png" width="250">



>* Understand implications of biases on society.
* Engage stakeholders to ensure fairness and equity.

>* AI models often lack transparency and accountability.
* Implement auditing to detect biases and explain decisions.

>* Developers must identify and mitigate biases proactively.
* Commit to ethical AI practices and collaboration.



## **2. Comprehensive Bias Mitigation Techniques**

### **2.1. Balancing Data and Ensuring Privacy**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20B/image_02_01.png" width="250">



>* Balancing data utility with privacy is crucial.
* Differential privacy protects individual data effectively.

>* Federated learning trains models without data exchange.
* Enhances privacy, reduces breaches, ensures compliance.

>* Synthetic data mimics real data properties.
* Promotes privacy and innovation in AI.



In [ ]:
#@title Python Code - Balancing Data and Ensuring Privacy

# The code demonstrates privacy-preserving AI techniques. It trains models on the Breast Cancer dataset using:
# 	1.	A baseline Logistic Regression model without privacy.
# 	2.	A differentially private Logistic Regression with diffprivlib.
# 	3.	Synthetic data generated to mimic real data for training.
# 	4.	Simulated federated learning, averaging weights from models trained on separate data “clients.”
# The goal is to balance data utility and privacy while comparing the accuracy of each approach.

# If running in Google Colab, install diffprivlib:
# !pip install diffprivlib
# Import libraries
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from diffprivlib.models import LogisticRegression as DPLogisticRegression
import warnings # Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# 1) Load the dataset
cancer = load_breast_cancer()
X_full = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y_full = pd.Series(cancer.target)

# Split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)

# A. Baseline Model (No Privacy)
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train, y_train)
baseline_preds = baseline_model.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_preds)

print("A) Baseline Logistic Regression (No Privacy):")
print(f"   Accuracy = {baseline_acc:.2f}\n")

# B. Differential Privacy with diffprivlib
dp_model = DPLogisticRegression(epsilon=1.0, data_norm=2.0, max_iter=1000, random_state=42)
dp_model.fit(X_train, y_train)
dp_preds = dp_model.predict(X_test)
dp_acc = accuracy_score(y_test, dp_preds)

print("B) Differentially Private Logistic Regression:")
print(f"   Epsilon = 1.0")
print(f"   Accuracy = {dp_acc:.2f}\n")

# C. Synthetic Data Generation
# This simple synthetic approach samples from a normal distribution
# using the mean and std of the real training set features.
# In practice, more advanced methods like GANs or other frameworks may be used.
synthetic_X_train = np.zeros_like(X_train)
for i in range(X_train.shape[1]):
    col_mean = X_train.iloc[:, i].mean()
    col_std = X_train.iloc[:, i].std()
    synthetic_X_train[:, i] = np.random.normal(loc=col_mean, scale=col_std, size=X_train.shape[0])

syn_model = LogisticRegression(max_iter=1000, random_state=42)
syn_model.fit(synthetic_X_train, y_train)
syn_preds = syn_model.predict(X_test)
syn_acc = accuracy_score(y_test, syn_preds)

print("C) Logistic Regression Trained on Synthetic Data:")
print("   (Features drawn from normal distribution approximating real data.)")
print(f"   Accuracy = {syn_acc:.2f}\n")

# D. Simulated Federated Learning
# We'll split the training set into N "clients" and train separate local models.
# Then we average their coefficients to simulate a simple federated approach.
# (In real scenarios, frameworks like TFF, Flower, or PySyft handle secure aggregation.)
def split_into_clients(X, y, n_clients=3):
    """Split (X, y) into n_clients sub-datasets."""
    X_clients = np.array_split(X, n_clients)
    y_clients = np.array_split(y, n_clients)
    return X_clients, y_clients

n_clients = 3
X_clients, y_clients = split_into_clients(X_train, y_train, n_clients=n_clients)

local_models = []
for i in range(n_clients):
    local_model = LogisticRegression(max_iter=1000, random_state=i)
    local_model.fit(X_clients[i], y_clients[i])
    local_models.append(local_model)

# Naive weight averaging (for illustration)
avg_coefs = np.mean([m.coef_ for m in local_models], axis=0)
avg_intercept = np.mean([m.intercept_ for m in local_models], axis=0)

# Perform a dummy fit with the entire training set to ensure at least 2 classes
fed_model = LogisticRegression(max_iter=1000, random_state=42)
fed_model.fit(X_train, y_train)  # ensures it sees both classe

# Overwrite the fitted coefficients with the aggregated (federated) ones
fed_model.coef_ = avg_coefs
fed_model.intercept_ = avg_intercept

fed_preds = fed_model.predict(X_test)
fed_acc = accuracy_score(y_test, fed_preds)

print("D) Simulated Federated Learning with 3 local clients:")
print(f"   Accuracy after naive weight averaging = {fed_acc:.2f}\n")

# Summary
print("SUMMARY OF RESULTS:")
print(f"   A) Baseline Model Accuracy:       {baseline_acc:.2f}")
print(f"   B) DP Model Accuracy (ε=1.0):     {dp_acc:.2f}")
print(f"   C) Synthetic Data Model Accuracy: {syn_acc:.2f}")
print(f"   D) Federated Learning Accuracy:   {fed_acc:.2f}")


### **2.2. Implementing Adversarial Debiasing Techniques**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20B/image_02_02.png" width="250">



>* Adversarial debiasing reduces bias in AI models.
* Focuses on relevant features, not sensitive attributes.

>* Balance model performance with fairness in debiasing.
* Adversarial networks reduce bias in credit scoring.

>* High computational cost and resource intensity.
* Combine with other strategies for comprehensive fairness.



In [ ]:
#@title Python Code - Implementing Adversarial Debiasing Techniques

# This code demonstrates adversarial debiasing using PyTorch.
# A main model learns to predict a target, while an adversary predicts a sensitive attribute from the main model’s representation.
# The main model is penalized for leaking sensitive information to the adversary.
# Training alternates between improving the adversary and updating the main model to balance accuracy and fairness.
# Results show the effectiveness of reducing bias while maintaining task performance.

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# 1. CREATE SYNTHETIC DATA
np.random.seed(42)
torch.manual_seed(42)

num_samples = 1000
num_features = 5

# Features
X = np.random.rand(num_samples, num_features).astype(np.float32)

# Sensitive attribute (binary: 0 or 1)
sensitive_attribute = np.random.randint(0, 2, size=num_samples)

# Target variable (binary) - artificially influenced by features + sensitive_attribute
# We add some correlation with the sensitive attribute to simulate bias.
Y = (
    X[:, 0]
    + 0.8 * X[:, 1]
    + 0.5 * sensitive_attribute
    + np.random.normal(0, 0.1, num_samples)
    > 1.5
).astype(np.int64)

# Convert everything to torch tensors
X_torch = torch.from_numpy(X)
sensitive_torch = torch.from_numpy(sensitive_attribute).long()
Y_torch = torch.from_numpy(Y)

# Split into train/test
X_train, X_test, s_train, s_test, y_train, y_test = train_test_split(
    X_torch, sensitive_torch, Y_torch, test_size=0.2, random_state=42
)

# 2. BUILD MAIN & ADVERSARY NNs
class MainModel(nn.Module):
    """
    The main model predicts the target (binary classification).
    We'll extract an intermediate 'representation' layer
    for the adversary to use.
    """
    def __init__(self, input_dim=5, hidden_dim=16):
        super(MainModel, self).__init__()
        self.rep = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU())
        self.classifier = nn.Linear(hidden_dim, 2)  # 2 classes for the target
    def forward(self, x):
        # Returns both the representation and the final logits
        representation = self.rep(x)
        logits = self.classifier(representation)
        return representation, logits

class Adversary(nn.Module):
    """
    The adversary tries to predict the sensitive attribute (binary).
    We'll take as input the representation from the main model.
    """
    def __init__(self, rep_dim=16, hidden_dim=8):
        super(Adversary, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(rep_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2)  # 2 classes for the sensitive attribute
        )
    def forward(self, rep):
        return self.net(rep)  # logits

# Instantiate the models
main_model = MainModel(input_dim=num_features, hidden_dim=16)
adversary = Adversary(rep_dim=16, hidden_dim=8)

# 3. SET UP OPTIMIZERS
# We'll create separate optimizers for the main model and the adversary.
main_optimizer = optim.Adam(main_model.parameters(), lr=0.01)
adv_optimizer = optim.Adam(adversary.parameters(), lr=0.01)

# 4. TRAINING CONFIGURATION
criterion_main = nn.CrossEntropyLoss()  # for main task (target)
criterion_adv = nn.CrossEntropyLoss()   # for adversary (sensitive attribute)

# How strongly to weigh the adversary's success (alpha).
# Higher alpha => stronger push to remove sensitive info from representation.
alpha = 1.0

# Training hyperparameters
epochs = 20
batch_size = 64

# We'll store losses for visualization
main_losses = []
adv_losses = []
adv_accs = []
main_accs = []

# Helper to compute accuracy
def get_accuracy(logits, labels):
    """Compute accuracy given logits (batch_size, num_classes) and true labels."""
    preds = torch.argmax(logits, dim=1)
    correct = (preds == labels).float().sum()
    return (correct / labels.shape[0]).item()

# 5. TRAINING LOOP
train_size = X_train.size(0)
num_batches = int(np.ceil(train_size / batch_size))

for epoch in range(epochs):
    epoch_main_loss = 0.0
    epoch_adv_loss = 0.0
    epoch_main_acc = 0.0
    epoch_adv_acc = 0.0

    # Shuffle indices for mini-batching
    indices = torch.randperm(train_size)

    for b in range(num_batches):
        start = b * batch_size
        end = start + batch_size
        batch_idx = indices[start:end]

        # Grab the batch
        x_batch = X_train[batch_idx]
        y_batch = y_train[batch_idx]
        s_batch = s_train[batch_idx]

        # PHASE 1: Train adversary
        # We freeze the main model and train the adversary to predict the sensitive attribute
        main_model.eval()
        adversary.train()

        adv_optimizer.zero_grad()

        with torch.no_grad():
            rep, _ = main_model(x_batch)

        adv_logits = adversary(rep)
        loss_adv = criterion_adv(adv_logits, s_batch)
        loss_adv.backward()
        adv_optimizer.step()

        # PHASE 2: Train main model
        # Now we freeze the adversary and update the main model to:
        # 1) Do well on the main classification task
        # 2) "Fool" the adversary (minimize -loss_adv => maximize loss_adv)
        main_model.train()
        adversary.eval()

        main_optimizer.zero_grad()

        rep, logits_main = main_model(x_batch)

        # Main task loss
        loss_main = criterion_main(logits_main, y_batch)

        # Adversary's prediction on the current rep
        with torch.no_grad():
            adv_logits_frozen = adversary(rep)

        # We want to maximize adv_loss => minimize (-adv_loss)
        loss_fool = -criterion_adv(adv_logits_frozen, s_batch)

        # Combined loss for the main model
        combined_loss = loss_main + alpha * loss_fool

        combined_loss.backward()
        main_optimizer.step()

        # Track losses & accuracy
        epoch_main_loss += loss_main.item() * x_batch.size(0)
        epoch_adv_loss += loss_adv.item() * x_batch.size(0)

        # Compute batch accuracy for main & adversary
        with torch.no_grad():
            main_acc = get_accuracy(logits_main, y_batch)
            epoch_main_acc += main_acc * x_batch.size(0)
            adv_acc = get_accuracy(adv_logits, s_batch)
            epoch_adv_acc += adv_acc * x_batch.size(0)

    # Average over epoch
    epoch_main_loss /= train_size
    epoch_adv_loss /= train_size
    epoch_main_acc /= train_size
    epoch_adv_acc /= train_size

    main_losses.append(epoch_main_loss)
    adv_losses.append(epoch_adv_loss)
    main_accs.append(epoch_main_acc)
    adv_accs.append(epoch_adv_acc)

    print(f"Epoch [{epoch+1}/{epochs}] | Main Loss: {epoch_main_loss:.3f} "
          f"| Adv Loss: {epoch_adv_loss:.3f} | Main Acc: {epoch_main_acc:.3f} "
          f"| Adv Acc: {epoch_adv_acc:.3f}")

# 6. EVALUATE ON TEST DATA
main_model.eval()
adversary.eval()

with torch.no_grad():
    rep_test, logits_test = main_model(X_test)
    test_main_acc = get_accuracy(logits_test, y_test)
    adv_logits_test = adversary(rep_test)
    test_adv_acc = get_accuracy(adv_logits_test, s_test)

print("\nFinal Evaluation on TEST Set:")
print(f" - Main Model Accuracy:       {test_main_acc:.3f}")
print(f" - Adversary (Sensitive) Acc: {test_adv_acc:.3f}")

# 7. OPTIONAL: PLOT TRAINING CURVES
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(main_losses, label='Main Loss')
axs[0].plot(adv_losses, label='Adversary Loss')
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Loss')
axs[0].legend()
axs[0].set_title('Loss Curves')

axs[1].plot(main_accs, label='Main Accuracy')
axs[1].plot(adv_accs, label='Adversary Accuracy')
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Accuracy')
axs[1].legend()
axs[1].set_title('Accuracy Curves')
plt.show()


### **2.3. Output Adjustment and Fairness Calibration**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20B/image_02_03.png" width="250">



>* Modify AI outputs for fairness criteria.
* Prevent biases in credit scoring and hiring.

>* Ensure predictions are accurate across groups.
* Monitor outputs to address disparities.

>* Balance fairness with model performance.
* Healthcare example: equitable treatment and accuracy.



In [ ]:
#@title Python Code - Output Adjustment and Fairness Calibration

# he code trains a Random Forest classifier on the Adult Income dataset to predict if an individual’s income exceeds $50K/year.
# It evaluates fairness by comparing True Positive Rate (TPR) and False Positive Rate (FPR) for males and females.
# A threshold adjustment is applied to improve fairness for females by lowering their decision threshold.
# Post-adjustment, the model’s fairness and overall performance metrics are re-evaluated to assess the impact.
# The goal is to demonstrate output adjustment and fairness calibration in AI models.

# 1. Install and import necessary libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ------------------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------------------
def compute_group_fairness_metrics(y_true, y_pred, group_membership):
    """
    Computes TPR (True Positive Rate) and FPR (False Positive Rate)
    for each group in the 'group_membership' array.
    """
    metrics_dict = {}
    unique_groups = np.unique(group_membership)
    for grp in unique_groups:
        grp_indices = (group_membership == grp)
        y_true_grp = y_true[grp_indices]
        y_pred_grp = y_pred[grp_indices]
        # Confusion matrix for this group
        tn, fp, fn, tp = confusion_matrix(y_true_grp, y_pred_grp, labels=[0, 1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        metrics_dict[grp] = {'TPR': tpr, 'FPR': fpr}
    return metrics_dict

def print_group_metrics(metrics_dict):
    """
    Prints TPR and FPR for each group in a readable format.
    """
    for grp, metrics in metrics_dict.items():
        print(f" Group: {grp} | TPR: {metrics['TPR']:.3f}, FPR: {metrics['FPR']:.3f}")

# ------------------------------------------------------------------------------
# 2. Load the Adult Income dataset
# ------------------------------------------------------------------------------
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
           'marital-status', 'occupation', 'relationship', 'race', 'sex',
           'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
df = pd.read_csv(url, header=None, names=columns, na_values=' ?', skipinitialspace=True)

# Remove rows with missing values
df.dropna(inplace=True)

# ------------------------------------------------------------------------------
# 3. Encode categorical features
# ------------------------------------------------------------------------------
label_encoders = {}
for col in df.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# 'income' is now binary-encoded: let's rename 1 -> High income, 0 -> Low income (for clarity)
# But for internal computation, we keep them as 0 and 1.
X = df.drop('income', axis=1)
y = df['income']

# We consider 'sex' as a protected attribute: 0->Female, 1->Male (after LabelEncoder)
protected_attr = X['sex'].values

# ------------------------------------------------------------------------------
# 4. Split the data and train a Random Forest classifier
# ------------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ------------------------------------------------------------------------------
# 5. Make predictions and evaluate overall performance
# ------------------------------------------------------------------------------
y_pred = model.predict(X_test)
print("=== Initial Model Evaluation ===")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ------------------------------------------------------------------------------
# 6. Evaluate group fairness metrics before threshold adjustment
# ------------------------------------------------------------------------------
print("=== Fairness Metrics Before Threshold Adjustment ===")
metrics_before = compute_group_fairness_metrics(y_test.values, y_pred, protected_attr[X_test.index])
print_group_metrics(metrics_before)

# ------------------------------------------------------------------------------
# 7. Threshold adjustment for one subgroup (e.g., Female = 0)
#    We'll apply a lower threshold to the probabilities for females to
#    encourage more positive predictions where the model might underpredict.
# ------------------------------------------------------------------------------
probs = model.predict_proba(X_test)[:, 1]

# Let's define different thresholds for female (grp=0) and male (grp=1).
threshold_female = 0.40
threshold_male   = 0.50

y_pred_adjusted = []
for i, p in enumerate(probs):
    if protected_attr[X_test.index][i] == 0:  # Female
        y_pred_adjusted.append(1 if p >= threshold_female else 0)
    else:  # Male
        y_pred_adjusted.append(1 if p >= threshold_male else 0)

y_pred_adjusted = np.array(y_pred_adjusted)

# ------------------------------------------------------------------------------
# 8. Evaluate model performance after threshold adjustment
# ------------------------------------------------------------------------------
print("\n=== Adjusted Model Evaluation ===")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_adjusted))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_adjusted))

# ------------------------------------------------------------------------------
# 9. Evaluate group fairness metrics after threshold adjustment
# ------------------------------------------------------------------------------
print("=== Fairness Metrics After Threshold Adjustment ===")
metrics_after = compute_group_fairness_metrics(y_test.values, y_pred_adjusted, protected_attr[X_test.index])
print_group_metrics(metrics_after)


### **2.4. Automated Monitoring Systems for Bias**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20B/image_02_04.png" width="250">



>* Continuous evaluation of AI models for bias.
* Proactive bias detection prevents reputational damage.

>* Scalable analysis across multiple AI models.
* Adapts to emerging biases dynamically.

>* False positives disrupt AI model functioning.
* High investment needed for implementation.



### **2.5. Collaborative Governance and Ethical Review**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20B/image_02_05.png" width="250">



>* Integrate diverse stakeholders for ethical AI development.
* Identify biases early through multi-disciplinary dialogue.

>* Ethical reviews assess AI fairness and bias.
* GDPR mandates DPIAs for high-risk technologies.

>* Diverse perspectives ensure equitable AI outcomes.
* Robust governance frameworks address bias and fairness.



## **3. Exploring Bias and Fairness in AI: A Real-World Case Study**

### **3.1. Understanding the AI System's Foundation**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20B/image_03_01.png" width="250">



>* AI systems rely on algorithmic data processing.
* Scrutinize data sources to prevent bias perpetuation.

>* AI architecture affects bias manifestation.
* Feature choice impacts bias and fairness.

>* Define fairness objectives for AI systems.
* Incorporate diverse perspectives in design phase.



### **3.2. Identifying Biases and Impacted Demographics**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20B/image_03_02.png" width="250">



>* AI biases threaten fairness and equity.
* Facial recognition errors highlight racial biases.

>* Analyze data for representational imbalances.
* Use fairness-aware preprocessing and auditing techniques.

>* Understand socio-cultural context of AI systems.
* Engage communities for inclusive AI development.



In [ ]:
#@title Python Code - Identifying Biases and Impacted Demographics

# The code uses the Adult Income dataset to train a Random Forest classifier to predict if income exceeds $50K based on demographic features.
# It preprocesses the data, encodes categorical variables, and splits it into training and testing sets.
# After training the model, it evaluates its performance and calculates prediction rates for different gender and race groups to detect potential bias in the model’s outcomes.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Load the Adult Income dataset from the UCI Machine Learning Repository
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

df = pd.read_csv(url, header=None, names=columns, na_values=' ?', skipinitialspace=True)

# Drop rows with missing values
df.dropna(inplace=True)

# Convert categorical variables to numerical using one-hot encoding
df = pd.get_dummies(
    df,
    columns=[
        'workclass', 'education', 'marital-status', 'occupation',
        'relationship', 'race', 'sex', 'native-country'
    ],
    drop_first=True
)

# Define features (X) and target variable (Y)
X = df.drop('income', axis=1)
Y = df['income'].apply(lambda x: 1 if x == '>50K' else 0)  # Convert income to binary (1 = >50K, 0 = <=50K)

# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

# Initialize and train a Random Forest classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, Y_train)

# Make predictions on the test set
Y_pred = model.predict(X_test)

# Evaluate the model's overall performance
print('Confusion Matrix:')
print(confusion_matrix(Y_test, Y_pred))
print('\nClassification Report:')
print(classification_report(Y_test, Y_pred))

# Create a DataFrame to analyze bias in predictions
predictions = pd.DataFrame({
    'actual': Y_test,
    'predicted': Y_pred
})

# Retrieve gender and race columns from the original dataframe
# Note: 'sex_Male' and 'race_White' are among the one-hot encoded columns if they exist
predictions['gender'] = df.loc[Y_test.index, 'sex_Male']  # 1 = Male, 0 = Female
predictions['race'] = df.loc[Y_test.index, 'race_White']  # 1 = White, 0 = Non-White

# Calculate mean prediction rates by gender
gender_bias = predictions.groupby('gender')['predicted'].mean()
print('\nPrediction rates by gender (1 = Male, 0 = Female):')
print(gender_bias)

# Calculate mean prediction rates by race
race_bias = predictions.groupby('race')['predicted'].mean()
print('\nPrediction rates by race (1 = White, 0 = Non-White):')
print(race_bias)


### **3.3. Implementing Solutions and Continuous Refinement**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20B/image_03_03.png" width="250">



>* Use algorithmic fairness techniques to reduce bias.
* Continuously refine solutions for evolving societal norms.

>* Form cross-functional bias mitigation teams.
* Mandate transparency and accountability in AI.

>* Set up feedback loops for ongoing analysis.
* Engage stakeholders for continuous improvement insights.



In [ ]:
#@title Python Code - Implementing Solutions and Continuous Refinement

# In this example, we:
# 1. Generate a synthetic dataset for credit-scoring with built-in bias
#    against a 'minority group.'
# 2. Use a logistic probability model to assign the 'creditworthy' label
#    (target) in a way that systematically disadvantages the minority.
# 3. Train two Random Forest models:
#    - Without bias mitigation
#    - With re-weighting (to give higher weight to minority samples)
# 4. Compare their performances on the test set using overall metrics
#    and group-specific metrics for minority vs. non-minority.

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# ----------------------
# 1. Data Generation
# ----------------------
np.random.seed(42)  # For reproducibility
num_samples = 2000
# Features:
#   - Age (integer from 18 to 69)
#   - Income (integer from 20,000 to 119,999)
#   - Minority_Group (binary: 0 = non-minority, 1 = minority)
age = np.random.randint(18, 70, size=num_samples)
income = np.random.randint(20000, 120000, size=num_samples)
minority_group = np.random.randint(0, 2, size=num_samples)

# Create a logistic function to determine creditworthiness probability
# z = alpha + beta_age * age + beta_inc * income + beta_minority * minority_group
# p = 1 / (1 + exp(-z))
alpha = -10.0
beta_age = 0.06
beta_inc = 0.00008
beta_minority = -2.0  # Negative coefficient to disadvantage the minority group

z = (alpha
     + beta_age * age
     + beta_inc * income
     + beta_minority * minority_group)
p = 1 / (1 + np.exp(-z))

# Sample the target from a Bernoulli distribution
target = np.random.binomial(n=1, p=p, size=num_samples)

# Construct DataFrame
data = pd.DataFrame({
    'Age': age,
    'Income': income,
    'Minority_Group': minority_group,
    'Creditworthy': target
})

# ----------------------
# 2. Train-Test Split
# ----------------------
X = data[['Age', 'Income', 'Minority_Group']]
y = data['Creditworthy']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ----------------------
# 3. Bias Mitigation
# ----------------------
# We'll train two models:
# (a) Without bias mitigation
# (b) With re-weighting (giving higher weight to minority samples)
# (a) No mitigation
clf_no_mitigation = RandomForestClassifier(random_state=42)
clf_no_mitigation.fit(X_train, y_train)

# (b) With re-weighting
weights = np.where(X_train['Minority_Group'] == 1, 3, 1)  # 3x weight for minority
clf_with_mitigation = RandomForestClassifier(random_state=42)
clf_with_mitigation.fit(X_train, y_train, sample_weight=weights)

# ----------------------
# 4. Predictions
# ----------------------
y_pred_no_mitigation = clf_no_mitigation.predict(X_test)
y_pred_with_mitigation = clf_with_mitigation.predict(X_test)

# ----------------------
# 5. Evaluation - Overall
# ----------------------
print("=== Overall Classification Report WITHOUT Bias Mitigation ===")
print(classification_report(y_test, y_pred_no_mitigation))
print("Accuracy:", accuracy_score(y_test, y_pred_no_mitigation))

print("\n=== Overall Classification Report WITH Bias Mitigation ===")
print(classification_report(y_test, y_pred_with_mitigation))
print("Accuracy:", accuracy_score(y_test, y_pred_with_mitigation))

# ----------------------
# 6. Evaluation - Group-Level
# ----------------------
def evaluate_by_group(y_true, y_pred, X_test, group_label=1):
    """Evaluate model performance for a specific group."""
    mask = (X_test['Minority_Group'] == group_label)
    group_name = "Minority" if group_label == 1 else "Non-Minority"
    print(f"\n--- Performance for {group_name} Group ---")
    print(classification_report(y_true[mask], y_pred[mask]))

print("\n=== Group-Level Evaluation: WITHOUT Bias Mitigation ===")
evaluate_by_group(y_test, y_pred_no_mitigation, X_test, group_label=0)
evaluate_by_group(y_test, y_pred_no_mitigation, X_test, group_label=1)

print("\n=== Group-Level Evaluation: WITH Bias Mitigation ===")
evaluate_by_group(y_test, y_pred_with_mitigation, X_test, group_label=0)
evaluate_by_group(y_test, y_pred_with_mitigation, X_test, group_label=1)


### **3.4. Community Engagement and Response**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20B/image_03_04.png" width="250">



>* Community engagement addresses AI biases effectively.
* Collaboration fosters trust and transparency in AI.

>* Community feedback enhances AI effectiveness and equity.
* Engagement leads to inclusive, socially responsible AI.

>* Continuous dialogue educates developers and communities.
* Regular meetings strengthen AI-community relationships.



### **3.5. Continuous Evaluation and Insights Gained**

<img src="https://storage.googleapis.com/mrafiei-odu/module_04/Lecture%20B/image_03_05.png" width="250">



>* Regularly assess AI model performance.
* Detect and adjust for new biases.

>* Identify and correct biases in AI systems.
* Enhance accuracy and ensure equitable predictions.

>* Continuous evaluation uncovers unexpected biases.
* Adjust strategies for inclusive marketing.



# <font color="#418FDE" size="6.5" uppercase>**Bias Mitigation Techniques and Case Study**</font>


In this lecture, you learned to:
- Apply bias detection and mitigation strategies to AI models.
- Analyze leading practices and trade-offs in promoting fairness.
- Examine a real-world bias and fairness case study.

In the next Module (Module 5), we will go over 'AI Risk Management'